In [3]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report, accuracy_score
import numpy as np

In [4]:
df = pd.read_csv("insurance.csv")

In [5]:
df.sample(5)

,age,weight,height,income_lpa,smoker,city,occupation,insurance_premium_category
68,20,80.3,1.87,0.68,False,Lucknow,student,Low
61,32,102.4,1.68,24.05,True,Chandigarh,unemployed,High
16,66,70.2,1.59,0.61,False,Pune,retired,Medium
20,34,58.2,1.85,30.65,True,Gaya,business_owner,Medium
89,24,108.8,1.89,4.00,False,Chennai,student,Low


In [8]:
df['occupation'].unique()

<StringArray>
[       'retired',     'freelancer',        'student', 'government_job',
 'business_owner',     'unemployed',    'private_job']
Length: 7, dtype: str

In [9]:
df_feat = df.copy()

In [10]:
# 1. BMI
df_feat['bmi'] = df_feat['weight']/(df_feat['height']**2)

In [11]:
# 2. Age group
def age_group(age):
    if age < 25:
        return "young"
    elif age < 45:
        return "adult"
    elif age < 60:
        return "middle_aged"
    return "senior"

In [22]:
df_feat["age_group"] = df_feat["age"].apply(age_group)

In [23]:
# 3. Lifestyle Risk
def lifestyle_risk(row):
    if row["smoker"] and row["bmi"] > 30:
        return "high"
    elif row["smoker"] and row["bmi"] > 27:
        return "medium"
    else:
        return "low"

In [24]:
df_feat["lifestyle_risk"] = df_feat.apply(lifestyle_risk, axis = 1)

In [25]:
tier_1_cities = ["Mumbai", "Delhi", "Bangalore", "Chennai", "Kolkata", "Hyderabad", "Pune"]
tier_2_cities = [
    "Jaipur", "Chandigarh", "Indore", "Lucknow", "Patna", "Ranchi", "Visakhapatnam", "Coimbatore",
    "Bhopal", "Nagpur", "Vadodara", "Surat", "Rajkot", "Jodhpur", "Raipur", "Amritsar", "Varanasi",
    "Agra", "Dehradun", "Mysore", "Jabalpur", "Guwahati", "Thiruvananthapuram", "Ludhiana", "Nashik",
    "Allahabad", "Udaipur", "Aurangabad", "Hubli", "Belgaum", "Salem", "Vijayawada", "Tiruchirappalli",
    "Bhavnagar", "Gwalior", "Dhanbad", "Bareilly", "Aligarh", "Gaya", "Kozhikode", "Warangal",
    "Kolhapur", "Bilaspur", "Jalandhar", "Noida", "Guntur", "Asansol", "Siliguri"
]

In [26]:
# 4. City Tier
def city_tier(city):
    if city in tier_1_cities:
        return 1
    elif city in tier_2_cities:
        return 2
    else:
        return 3

In [27]:
df_feat["city_tier"] = df_feat["city"].apply(city_tier)

In [28]:
df_feat.drop(columns=['age', 'weight', 'height', 'smoker', 'city'])[['income_lpa', 'occupation', 'bmi', 'age_group', 'lifestyle_risk', 'city_tier', 'insurance_premium_category']].sample(5)
     

,income_lpa,occupation,bmi,age_group,lifestyle_risk,city_tier,insurance_premium_category
74,1.880000,retired,29.979760,senior,medium,2,High
91,28.467885,government_job,38.675103,adult,low,1,Low
15,2.990000,retired,21.860828,senior,low,1,Medium
98,28.300000,business_owner,30.521676,adult,low,1,Low
19,2.790000,student,43.437500,young,high,2,High


In [ ]:
X = df_feat[["bmi", "age_group", "lifestyle_risk", "city_tier", "income_lpa", "occupation"]]
Y = df_feat["insurance_premium_category"]
